### Making and training the model

Good things to have for ML training:

- saved checkpoints (loss history, current epoch)
- resume training
- early stopping (with or without patience counter)
- load weights mechanism

Perhaps have:
- stop training mechanism 
- early sanity checks
- memory management

In [1]:
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from datetime import datetime 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Various functions

In [ ]:
def train(model, training_dl, validation_dl, device,
          epoch_start, epochs, 
          optimizer, loss_function,
          filename,
          verbose=0,
          seed=42,
          training_history=None, validation_history=None):
    # THIS FUNCTION REFERENCES 'train' IN 'Object_localization_Kjetil_Jan' IN CELL 15 OF 32!

    assert epoch_start > 0
    assert epochs > 0

    torch.manual_seed(seed)
                        # Setting the seed for pytorch for reproducibility.

    if training_history is None:
        training_history = []
    if validation_history is None:
        validation_history = []

    model.to(device)    
                        # Moves the object to the hardware 'device'.

    n_training_batches = len(training_dl)       
                                                # len() returns the number of batches in the
                                                # dataloader dependent on batch size. There may
                                                # be a last batch with size smaller than the
                                                # given batch size.
    n_validation_batches = len(validation_dl)


    for epoch in range(epoch_start, epoch_start + epochs):
        # Training
        model.train()
                            # Sets the model in training mode and toggles 
                            # layer behaviour.

        training_loss = 0.0

        for features, labels in training_dl:
                                # Retrives a batch from the DataLoader containing
                                # the features and labels.

            features = features.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
                                # Resets the gradients (the .grad attribute)
                                # of the torch.Tensor-s.
            
            preds = model(features)     
                                # 'features' with a set batch size are vectorized
                                # and passed through the network in one operation.
                                # The ouput will be of dimensions (32, class_amount).

            loss = loss_function(preds, labels)     # Loss function calculates a mean loss of
                                                    # the multiple predictions and labels.

            loss.backward()
                                # PyTorch's AutoGrad performs backpropagation to
                                # update each trainable (requires_grad=True) tensor's 
                                # .grad attribute.

            optimizer.step()
                                # The optimizer (e.g. Adam or SGD) uses the gradients
                                # to adjust the models parameters.

            training_loss += loss.item()

        training_history.append(training_loss / n_training_batches)

        # Validation
        model.eval()
        validation_loss = 0.0

        with torch.no_grad():
                                # Disables gradient calculation.

            for features, labels in validation_dl:
                features = features.to(device)
                labels = labels.to(device)
                preds = model(features)
                loss = loss_function(preds, labels)
                validation_loss += loss.item()

        validation_history.append(validation_loss / n_validation_batches)

        # Progress report/verbosity
        if (verbose != 0 and (epoch - epoch_start) % verbose == 0):
            print('{} | Epoch {:3d} | Train Loss: {:.3f} | Val Loss: {:.3f}'.format(
                datetime.now().time(), epoch, training_history[-1], validation_history[-1]))

        # Saving checkpoint/best model/final model
        if (True):
            checkpoint = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'model_architecture': str(model),
                'manual_seed': seed,
                'training_history': training_history,
                'validation_history': validation_history

                # To ensure dataloaders shuffle state, might need to
                # save Pythons random-module state and NumPy's.
            }
            torch.save(checkpoint, r'./model_training/' + filename + '.pth')

    return training_history, validation_history

def performace_metric():
    pass

def load_weights():
    pass

def plot_loss():
    pass

### Simple CNN MNIST model

In [3]:
class MNISTmodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(3, 3), padding=1)
        self.flat = nn.Flatten()
        self.dense1 = nn.Linear(in_features=28*28, out_features=10)
        # self.sm = nn.Softmax(dim=1)

    def forward(self, x):
        out = self.conv1(x)
        out = self.flat(out)
        out = self.dense1(out)
        # out = self.sm(out)
        return out

# Test if output dims are correct
test_model = MNISTmodel()
test_input = torch.randn(1, 1, 28, 28)
test_output = test_model(test_input)
print(f'Test-output dimensions: {test_output.shape}')
del test_model, test_input, test_output

Test-output dimensions: torch.Size([1, 10])


### Training

In [4]:
transform = transforms.Compose([ transforms.ToTensor() ])
                    # TODO How it work?

full_training_data = datasets.MNIST(
    root='./data', 
    train=True, 
                        # Downloads the 60000 images used for training.
                        # If False, it will download the 10000 seperate
                        # ones used for testing.
    download=True, 
    transform=transform
)

training_size = 50000
validation_size = 10000

training_dataset, validation_dataset = random_split(full_training_data, [training_size, validation_size])

BATCH_SIZE = 64
training_dl = DataLoader(training_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_dl = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9912422/9912422 [00:05<00:00, 1671769.06it/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28881/28881 [00:00<00:00, 231602.83it/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1648877/1648877 [00:01<00:00, 1637859.54it/s]


Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4542/4542 [00:00<00:00, 925771.64it/s]

Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw



In [7]:
model = MNISTmodel()
optimizer = torch.optim.SGD(model.parameters(), lr=0.003)

training_history, validation_history = train(
    model, training_dl, validation_dl, device,
    1, 5,
    optimizer, F.cross_entropy,
    'checkpoint',
    verbose=1
    )

15:28:00.835755 | Epoch   1 | Train Loss: 1.046 | Val Loss: 0.485
15:28:09.952413 | Epoch   2 | Train Loss: 0.412 | Val Loss: 0.391
15:28:19.382871 | Epoch   3 | Train Loss: 0.361 | Val Loss: 0.361
15:28:29.130909 | Epoch   4 | Train Loss: 0.339 | Val Loss: 0.341
15:28:38.035135 | Epoch   5 | Train Loss: 0.326 | Val Loss: 0.332
